# CIFAR-10 CNN: Tucker-2圧縮後のfine-tuning

`02_rank_sweep.ipynb` で選んだTucker-2 rankを使い、
圧縮直後に落ちたaccuracyをfine-tuningでどこまで回復できるか確認する。

このNotebookの目的は、
**圧縮 -> 精度低下 -> fine-tuningで回復**
というNN圧縮の実験フローをTucker-2でも確認すること。


## 1. baseline・DataLoader・学習/評価関数を準備する

SVD実験と同様、既存の `src` の共通処理を優先して使う。

比較条件を揃えるため、

- dataset split
- transform
- batch size
- baseline checkpoint

を不用意に変更しない。


In [ ]:
# TODO:
# baseline model / checkpoint
# train / validation / test loaders
# criterion
# device
# evaluate / train_one_epoch または fit_with_early_stopping


## 2. rank sweepで選んだrankを固定する

`02_rank_sweep.ipynb` の結果から、fine-tuning対象を決める。

ここでは新しいrank探索をしない。


In [ ]:
rank_out = None  # TODO
rank_in = None   # TODO


## 3. baselineから圧縮モデルを作る

baselineをdeepcopyし、01で作ったTucker-2置換処理で対象Convを置き換える。

fine-tuning前の状態を必ず評価して保存する。
これが「圧縮だけでどれだけ精度が落ちたか」の基準になる。


In [ ]:
# TODO:
# compressed_model = ...
# pre_ft_loss, pre_ft_acc = evaluate(...)
# print(...)


## 4. fine-tuning条件を決める

最初は短いepoch数でよい。

確認するもの:

- optimizer
- learning rate
- epoch数
- early stoppingを使うか

baselineをゼロから再学習するのではなく、
**Tucker-2で初期化済みの圧縮モデルを微調整する**。


In [ ]:
# TODO:
# optimizer = ...
# epochs = ...


## 5. fine-tuningする

既存の `train_one_epoch` / `fit_with_early_stopping` が使えるなら再利用する。

各epochで少なくともvalidation accuracyを残し、
fine-tuning前後を比較できるようにする。


In [ ]:
history = []

# TODO:
# for epoch in range(...):
#     ...


## 6. fine-tuning後を評価する

最低限、次を並べる。

```text
baseline accuracy
Tucker-2直後 accuracy
fine-tuning後 accuracy

baseline parameters
Tucker-2 parameters
parameters reduction
```

必要なら推論時間やMACsも追加するが、
まずはパラメータ数とaccuracyの回復を優先する。


In [ ]:
# TODO:
# post_ft_loss, post_ft_acc = evaluate(...)
# 比較表を作る。


## 7. 結果を保存する

既存の `get_experiment_dirs` 等の保存方針に合わせて、
モデル・結果をTucker用のmethod/case/experiment配下へ保存する。

Notebook内に独自の保存先ルールを増やさない。


In [ ]:
# TODO:
# 既存の保存helperを使ってcheckpoint / csv等を保存する。


## 8. このNotebookの完了条件

次を確認できれば、1層Tucker-2の一連の実験は完了。

1. Tucker-2圧縮直後のaccuracy dropを測れた
2. fine-tuning後にaccuracyがどれだけ戻ったか測れた
3. baselineに対するパラメータ削減量を維持できている
4. rank・圧縮率・fine-tuning後accuracyの関係を説明できる

その次に進む候補:

- 複数Conv層のTucker-2
- model-wide rank設計
- HOOIによるfactor改善
- TT/MPSへの移行
